In [1]:
!pip install datasets transformers evaluate seqeval

In [2]:
from transformers import TrainingArguments

C:\Users\Samyagh\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    pipeline,
    DataCollatorForTokenClassification
)
import numpy as np
import evaluate

In [4]:
dataset = load_dataset("lhoestq/conll2003")

In [5]:
label_column = "pos_tags"

In [6]:
# label_column = "chunk_tags"

In [7]:
# Select task
label_column = "pos_tags"   # change to "chunk_tags" for chunking

# Define labels manually (based on dataset)
if label_column == "pos_tags":
    label_names = [
        '"', "''", '#', '$', '(', ')', ',', '.', ':', '``',
        'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS',
        'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT',
        'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM',
        'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ',
        'WDT', 'WP', 'WP$', 'WRB'
    ]

elif label_column == "chunk_tags":
    label_names = [
        'O','B-ADJP','I-ADJP','B-ADVP','I-ADVP','B-CONJP','I-CONJP',
        'B-INTJ','I-INTJ','B-LST','I-LST','B-NP','I-NP','B-PP',
        'I-PP','B-PRT','I-PRT','B-SBAR','I-SBAR','B-UCP','I-UCP',
        'B-VP','I-VP'
    ]

# Create mappings
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}
num_labels = len(label_names)

In [8]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [9]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",
        max_length=32, 
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples[label_column]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        prev = None

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != prev:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            prev = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

Map: 100%|████████████████████████████████████████████████████████████████| 3453/3453 [00:00<00:00, 6634.36 examples/s]


In [10]:
train_dataset = tokenized_dataset["train"].select(range(800))
eval_dataset = tokenized_dataset["validation"].select(range(200))

In [11]:
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

Loading weights: 100%|█████████████████████████████████████████████████████████████| 197/197 [00:00<00:00, 4023.32it/s]
BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading 

In [12]:
data_collator = DataCollatorForTokenClassification(tokenizer)

In [13]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="no",
    learning_rate=5e-5,
    num_train_epochs=1,
    per_device_train_batch_size=4
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator
)

trainer.train()

C:\Users\Samyagh\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.79it/s]


TrainOutput(global_step=200, training_loss=1.2845556640625, metrics={'train_runtime': 158.8275, 'train_samples_per_second': 5.037, 'train_steps_per_second': 1.259, 'total_flos': 13070153164800.0, 'train_loss': 1.2845556640625, 'epoch': 1.0})

In [15]:
import evaluate
import numpy as np

# Load metric
metric = evaluate.load("seqeval")

# Define compute_metrics
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id2label[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [id2label[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    return metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

# ✅ FINAL TRAINER (IMPORTANT FIX)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics   # 🔥 ADD HERE
)

# ✅ ONLY RUN THIS
trainer.train()

Step,Training Loss


Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.05it/s]


TrainOutput(global_step=200, training_loss=0.40630409240722654, metrics={'train_runtime': 157.3247, 'train_samples_per_second': 5.085, 'train_steps_per_second': 1.271, 'total_flos': 13070153164800.0, 'train_loss': 0.40630409240722654, 'epoch': 1.0})

In [16]:
nlp = pipeline("token-classification", model=model, tokenizer=tokenizer)

sentence = "John works at Google in California"

output = nlp(sentence)

for token in output:
    print(token["word"], token["entity"])

john NNP
works VBZ
at NNP
google NNP
in IN
california NNP
